# 01 — Dataset Creation

Builds the two datasets used in the thesis (Chapter 3):

| Output | Used by | Description |
|---|---|---|
| `data/processed/rebuilt/prescribed_treatment_data_rebuilt.csv` | 03, 05 | One row per CAL FIRE wildfire incident (2013–2022), with prior-month county climate, fire-day maximum wind, and prescribed-burn activity within 5/10/25/50/100 km over the previous 3/6/9/12/24/36 months |
| `data/processed/rebuilt/monthly_data_rebuilt.csv` | 02, 04 | Statewide monthly totals of wildfire and prescribed-fire acres plus statewide climate |

**This notebook never overwrites the canonical files** in `data/processed/`
(`prescribed_treatment_data_new_clean.csv`, `prescribed_treatment_data_new.csv`, `monthly_data.csv`).
The last section checks that the rebuild matches them. `max_wind` comes from the Open-Meteo archive and was
corrected in November 2026 (see the *Wind speed* section).

See `docs/data_dictionary.md` for column definitions.

### Changes from the 2024 version (original kept in `old/original_notebooks_2024/`)
* Paths updated to the new `data/raw/...` layout.
* The spatial-temporal prescribed-burn join was a Python double loop using `geopy` (≈5.5 h).
  It is now vectorized with `pyproj.Geod` (the same WGS-84 geodesic, Karney's algorithm, that
  `geopy.distance.distance` uses) and runs in about a minute. Set `USE_ORIGINAL_SLOW_LOOP = True` to run the original loop.
* Column names now get the underscore the canonical file has (`prescribed_3_months_...`,
  previously `prescribed3_months_...`), and `log_incident_acres_burned` is computed on the output table.
* Cells are in execution order, so *Restart & Run All* works.

In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
from pyproj import Geod
from tqdm import tqdm
plt.style.use("ggplot")

from functions import *   # convert_acres_degrees, path constants

# Settings
USE_ORIGINAL_SLOW_LOOP = False   # True -> original geopy double loop (~5.5 hours)
OUT_DIR = os.path.join("data", "processed", "rebuilt")
os.makedirs(OUT_DIR, exist_ok=True)

## Loading data
### CAL FIRE wildfire incidents
Source: `data/raw/calfire/mapdataall.csv` (downloaded in 2024 from https://incidents.fire.ca.gov/imapdata/mapdataall.csv).
Note: the live file keeps changing, so re-downloading it will not reproduce the thesis numbers exactly.

Cleaning steps (same as the thesis):
* use the creation date as the fire date (`ex_date`); the end dates are unreliable.
* keep incidents with latitude in (31, 43) and longitude in (-125, -114). Two hard-coded corrections:
  row 1736 had a bad longitude, and one longitude was recorded with the wrong sign.
* keep 2013-01-01 to 2022-12-31.

In [2]:
df = pd.read_csv("data/raw/calfire/mapdataall.csv")
df["acres_decimal"] = [convert_acres_degrees(df.at[i,"incident_acres_burned"], df.at[i, "incident_latitude"]) for i in range(len(df))]
df["incident_date_created"] = pd.to_datetime(df["incident_date_created"].fillna(df["incident_date_extinguished"]))
df["ex_date"] = pd.to_datetime(df["incident_date_created"])
df = df.sort_values('ex_date', ascending=True)
df = df.loc[(df["incident_latitude"]>31) &(df["incident_latitude"]<43)].reset_index(drop=True)
df.at[1736,"incident_longitude"] = -119.1414610
df.at[np.argmax(df["incident_longitude"]),"incident_longitude"] = - df.at[np.argmax(df["incident_longitude"]),"incident_longitude"]
df = df.loc[(df["incident_longitude"]>-125) &(df["incident_longitude"]<-114)].reset_index(drop=True)

df = df.set_index("ex_date")
df = df.loc['2013-01-01':'2022-12-31']
wildfire_monthly_acres_df = df[["incident_acres_burned"]].resample('M').sum()
df["year"] = df.index.year
df["month"] = df.index.month
print(len(df), "wildfire incidents 2013-2022")

2044 wildfire incidents 2013-2022


### Prescribed burns
Source: `data/raw/prescribed_burns/California_Fire_Perimeters_(all).geojson` (CAL FIRE prescribed burn / fuel
treatment perimeters, https://data.ca.gov/dataset/prescribed-burns). Each burn is represented by the
centroid of its polygon (`lon_center`, `lat_center`) and its treated acres (`TREATED_AC`).

In [3]:
gdf = gpd.read_file("data/raw/prescribed_burns/California_Fire_Perimeters_(all).geojson")
gdf["lon_center"] = gdf["geometry"].centroid.x   # geographic-CRS centroid (warning is expected, error is negligible at this scale)
gdf["lat_center"] = gdf["geometry"].centroid.y
gdf["acres_decimal"] = [convert_acres_degrees(gdf.at[i,"TREATED_AC"], gdf.at[i, "lat_center"]) for i in range(len(gdf))]

gdf = gdf.fillna(0)
gdf["START_DATE"] = pd.to_datetime(gdf["START_DATE"], errors = 'coerce')
gdf = gdf.sort_values('START_DATE', ascending=True)
gdf = gdf.set_index("START_DATE")
gdf_mon = gdf[["TREATED_AC"]].resample('M').sum()
gdf_mon = gdf_mon.loc['2011-01-01':'2022-12-31']

/var/folders/cn/5p5n6ztx4310_zhlpq83ldym0000gn/T/ipykernel_42143/700633518.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["lon_center"] = gdf["geometry"].centroid.x   # geographic-CRS centroid (warning is expected, error is negligible at this scale)
/var/folders/cn/5p5n6ztx4310_zhlpq83ldym0000gn/T/ipykernel_42143/700633518.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["lat_center"] = gdf["geometry"].centroid.y


In [4]:
# Statewide monthly prescribed acres in the previous 3/6/9/12 months (for the monthly dataset).
# Note: these slices include the month 4/7/10/13 months back through the previous month, because
# .loc slicing on dates includes both ends.
gdf_mon["acres_3_month"] = [gdf_mon.loc[i - pd.DateOffset(months=4):i- pd.DateOffset(months=1)]["TREATED_AC"].sum() for i in gdf_mon.index]
gdf_mon["acres_6_month"] = [gdf_mon.loc[i - pd.DateOffset(months=7):i- pd.DateOffset(months=1)]["TREATED_AC"].sum() for i in gdf_mon.index]
gdf_mon["acres_9_month"] = [gdf_mon.loc[i - pd.DateOffset(months=10):i- pd.DateOffset(months=1)]["TREATED_AC"].sum() for i in gdf_mon.index]
gdf_mon["acres_12_month"] = [gdf_mon.loc[i - pd.DateOffset(months=13):i- pd.DateOffset(months=1)]["TREATED_AC"].sum() for i in gdf_mon.index]
prescribed_monthly = gdf_mon.loc['2013-01-01':'2022-12-31']

## Climate data
NOAA NCEI *Climate at a Glance* monthly series
(https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/).
* `data/raw/noaa_climate/monthly_climate_data.csv`: **statewide** California values per month.
* `data/raw/noaa_climate/climate_data.xlsx`: **county** values, one sheet per variable
  (`tavg, tmax, tmin, pcp, cdd, hdd, pdsi, phdi, pmdi, zndx`), one column per county.

Each wildfire gets the value for the **calendar month before the fire**, averaged over its listed
counties. If a fire has no county listed, it gets the statewide value instead.

In [5]:
def diff_month(d1, d2):
    """Number of calendar months from d2 to d1 (e.g. 2020-03-31 vs 2020-01-01 -> 2)."""
    return (d1.year - d2.year) * 12 + d1.month - d2.month

monthly_climate = pd.read_csv("data/raw/noaa_climate/monthly_climate_data.csv", index_col=0)
monthly_climate.index = pd.to_datetime(monthly_climate.index)
monthly_climate = monthly_climate.loc['2013-01-01':'2022-12-31']
monthly_climate.index = pd.to_datetime(monthly_climate.index).date

# Statewide monthly dataset (used in 02 and 04)
# put all three monthly tables on the same (UTC, month-end) index before joining
def _month_end_utc(index):
    idx = pd.DatetimeIndex(pd.to_datetime(index))
    idx = idx.tz_localize("UTC") if idx.tz is None else idx.tz_convert("UTC")
    return idx.normalize()
_parts = []
for _t in (prescribed_monthly, wildfire_monthly_acres_df, monthly_climate):
    _t = _t.copy(); _t.index = _month_end_utc(_t.index); _parts.append(_t)
monthly_fires = pd.concat(_parts, axis=1)
monthly_fires = monthly_fires.rename(columns={"TREATED_AC": "rx_acres", "acres_3_month": "rx_acres_3_month",
                                              "acres_6_month": "rx_acres_6_month", "acres_9_month": "rx_acres_9_month",
                                              "acres_12_month": "rx_acres_12_month"})
monthly_fires.fillna(0).to_csv(os.path.join(OUT_DIR, "monthly_data_rebuilt.csv"))

In [6]:
climate_dfs = pd.read_excel("data/raw/noaa_climate/climate_data.xlsx", sheet_name=None, index_col=0)
for sheet in climate_dfs:
    climate_dfs[sheet].index = pd.to_datetime(climate_dfs[sheet].index)
print(list(climate_dfs))

['tavg', 'tmax', 'tmin', 'pcp', 'cdd', 'hdd', 'pdsi', 'phdi', 'pmdi', 'zndx']


In [7]:
import warnings
climate_data = {sheet: [] for sheet in climate_dfs}
df = df.reset_index(drop=False)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # "mean of empty slice" for counties missing from the sheet -> NaN
    for sheet in climate_dfs:
        current_climate = climate_dfs[sheet]
        for i in range(len(df)):
            fire_date = df.at[i,"ex_date"]
            county = df.at[i, "incident_county"]
            if not pd.isna(county):
                counties = county.split(", ")
                inds = [k for k in range(len(current_climate)) if (diff_month(fire_date, current_climate.index[k])<=1) and (diff_month(fire_date, current_climate.index[k])>0)]
                climate_for_fire = current_climate.iloc[inds][:].reset_index(drop=True)
                county_cols = [col for col in climate_for_fire.columns if " ".join(col.split(" County")[:-1]) in counties]
                climate_vals = [climate_for_fire.at[0,county_col] for county_col in county_cols]
                climate_data[sheet].append(np.mean(climate_vals))
            else:
                # no county listed -> statewide value for the previous month
                inds = [k for k in range(len(monthly_climate)) if (diff_month(fire_date, monthly_climate.index[k])<=1) and (diff_month(fire_date, monthly_climate.index[k])>0)]
                climate_val = monthly_climate.iloc[inds][:].reset_index(drop=True).at[0, sheet]
                climate_data[sheet].append(climate_val)
df = pd.concat([df, pd.DataFrame(climate_data)], axis=1)

## Prescribed burns near each wildfire (spatial-temporal join)
For each wildfire, for each look-back window **T ∈ {3, 6, 9, 12, 24, 36} months** and radius
**R ∈ {5, 10, 25, 50, 100} km**, a prescribed burn counts when

* `0 < diff_month(fire_date, burn_start_date) <= T` (it started in a calendar month before the fire's month and at most T months earlier), and
* the geodesic (WGS-84) distance from the wildfire's reported point to the burn polygon's centroid is ≤ R.

`prescribed_{T}_months_in_{R}km` is True if at least one burn counts, and `prescribed_{T}_months_size_{R}km`
is the sum of `TREATED_AC` over those burns.

In [8]:
# correcting incorrect lat/lons (kept from the original notebook)
df = df.loc[(df["incident_latitude"]>-90) &(df["incident_latitude"]<90)].reset_index(drop=True)
df.at[np.argmin(df["incident_longitude"]),"incident_longitude"] = -119.1414610 # fix incorrect long

gdf = gdf.sort_values('START_DATE', ascending=True)
gdf = gdf.loc['2010-01-01':'2023-12-31']
gdf = gdf.reset_index()

times = [3, 6, 9, 12, 24, 36]
radii = [5, 10, 25, 50, 100]
keep_cols = ['ex_date', 'incident_name', 'incident_county', 'incident_acres_burned', 'incident_longitude','incident_latitude', 'incident_type',
             "tavg", "tmax", "tmin", "pcp", "cdd", "hdd", "pdsi", "phdi", "pmdi", "zndx"]
tdf = df[keep_cols].copy()

In [9]:
if not USE_ORIGINAL_SLOW_LOOP:
    # Vectorized version: month-difference matrix (fires x burns) and geodesic distance matrix in km.
    fire_m = (tdf["ex_date"].dt.year * 12 + tdf["ex_date"].dt.month).to_numpy()[:, None]
    burn_m = (gdf["START_DATE"].dt.year * 12 + gdf["START_DATE"].dt.month).to_numpy()[None, :]
    month_diff = fire_m - burn_m
    geod = Geod(ellps="WGS84")
    n_f, n_b = len(tdf), len(gdf)
    flon = np.repeat(tdf["incident_longitude"].to_numpy(float), n_b)
    flat = np.repeat(tdf["incident_latitude"].to_numpy(float), n_b)
    blon = np.tile(gdf["lon_center"].to_numpy(float), n_f)
    blat = np.tile(gdf["lat_center"].to_numpy(float), n_f)
    _, _, dist_m = geod.inv(flon, flat, blon, blat)
    dist_km = (dist_m / 1000).reshape(n_f, n_b)
    acres = gdf["TREATED_AC"].to_numpy(float)[None, :]
    new_cols = {}
    for time in times:
        in_window = (month_diff > 0) & (month_diff <= time)
        for r in radii:
            mask = in_window & (dist_km <= r)
            new_cols[f"prescribed_{time}_months_in_{r}km"] = mask.any(axis=1)
            new_cols[f"prescribed_{time}_months_size_{r}km"] = (mask * acres).sum(axis=1)
    tdf = pd.concat([tdf, pd.DataFrame(new_cols, index=tdf.index)], axis=1)
else:
    # Original implementation (≈5.5 hours).
    from geopy import distance
    for time in tqdm(times):
        res = {f"{k}_{r}km": [] for r in radii for k in ("in", "size")}
        for i in tqdm(range(len(tdf))):
            lat = tdf.iloc[i]["incident_latitude"]; lon = tdf.iloc[i]["incident_longitude"]; df_date = tdf.iloc[i]["ex_date"]
            inds = [k for k in range(len(gdf)) if (diff_month(df_date, gdf.iloc[k]["START_DATE"])<=time) and (diff_month(df_date,gdf.iloc[k]["START_DATE"])>0)]
            gdf_t = gdf.iloc[inds][:].reset_index(drop=True)
            flags = {r: False for r in radii}; sizes = {r: 0 for r in radii}
            for j in range(len(gdf_t)):
                dist = distance.distance((lat,lon), (gdf_t.iloc[j]["lat_center"], gdf_t.iloc[j]["lon_center"])).km
                for r in radii:
                    if dist <= r:
                        flags[r] = True; sizes[r] += gdf_t.iloc[j]["TREATED_AC"]
            for r in radii:
                res[f"in_{r}km"].append(flags[r]); res[f"size_{r}km"].append(sizes[r])
        res = {f"prescribed_{time}_months_{key}": value for key, value in res.items()}
        tdf = pd.concat([tdf, pd.DataFrame(res)], axis=1)
# column order: for each time window, (in, size) per radius, as in the thesis file
order = keep_cols + [f"prescribed_{t}_months_{k}_{r}km" for t in times for r in radii for k in ("in", "size")]
tdf = tdf[order]
tdf.shape

(2044, 77)

## County, region and log target
* `county`: county whose boundary polygon contains the wildfire point ("none" if outside California).
* `incident_county`: missing values are filled from `county`.
* `county_new`: `incident_county`, or `"multiple"` when the incident lists more than one county.
* `region`: one of 8 regions, assigned from `county_new` (mapping below).

In [10]:
tdf["log_incident_acres_burned"] = np.log(tdf['incident_acres_burned'].replace(0, np.nan)).replace(np.nan, 0)

county_boundary = gpd.read_file("data/raw/boundaries/California_County_Boundaries.geojson")
tdf["county"] = [county_boundary.loc[county_boundary["geometry"].contains(Point(tdf.at[i,"incident_longitude"], tdf.at[i,"incident_latitude"]))].reset_index(drop=True).at[0,"COUNTY_NAME"]
                if len(county_boundary.loc[county_boundary["geometry"].contains(Point(tdf.at[i,"incident_longitude"], tdf.at[i,"incident_latitude"]))]) > 0
                else "none"
                for i in range(len(tdf))]
tdf["incident_county"] = tdf["incident_county"].fillna(tdf["county"])

county_regions = {
    "North Coast": ['Del Norte', 'Humboldt', 'Mendocino', "Sonoma", "Lake", "Napa", "Marin", "Trinity"],
    "Shasta Cascades": ['Siskiyou', "Trinity", "Shasta", "Lassen", "Modoc",
                        'Tehama', "Butte", "Plumas", "State of Oregon"],
    "Central Valley": ["Glenn", "Colusa", "Sutter","Yolo", "Sacramento", "Solano", "Fresno", "Kern", "Kings",
                       "Madera", "Merced", "Placer", "San Joaquin", "Tehama", "Yuba", "Stanislaus"],
    "Sierras": ["Alpine", "Amador", "Calaveras", "El Dorado", "Inyo", "Mariposa", "Mono", "Nevada", "Plumas", "Sierra", "Tulare",
                "Tuolumne", "State of Nevada"],
    "Central Coast": ["Contra Costa", "Alameda", "San Fransisco", "San Mateo", "Santa Clara",
                      "Santa Cruz", "San Benito", "Monterey", "San Luis Obispo", "Santa Barbara", "Ventura"],
    "South Coast": ["Los Angeles", "Orange", "San Diego"],
    "Desert": ["Riverside", "San Bernardino", "Imperial", "Mexico"],
    "Multiple": ["multiple"]
}
# NB: a county listed in two regions (Trinity, Tehama, Plumas) is assigned to the LAST one in this dict
# (Shasta Cascades, Central Valley, Sierras respectively), because of how the dict below is built.
regions_per_county = {v: k for k, val in county_regions.items() for v in val}

tdf["county_new"] = ["multiple" if len(str(county).split(","))>1 else county for county in tdf["incident_county"]]
tdf = tdf.loc[tdf["county"]!="none"].reset_index(drop=True)
tdf["region"] = [regions_per_county.get(county) for county in tdf["county_new"].to_list()]

## Final cleaning and save
* drop fires with missing acres
* drop fires outside California (`county == "none"`; e.g. reported in Mexico)

In [11]:
tdf = tdf.dropna(subset="incident_acres_burned").reset_index(drop=True)
tdf = tdf.loc[tdf["county"]!="none"].reset_index(drop=True)
tdf.to_csv(os.path.join(OUT_DIR, "prescribed_treatment_data_rebuilt.csv"))
tdf.shape

(1992, 81)

## Wind speed (`max_wind`)
`max_wind` = **maximum 10 m wind speed (km/h) on the local (Pacific) calendar day the fire started**, at the
reported fire point, from the Open-Meteo historical weather archive (ERA5-based reanalysis;
https://open-meteo.com/en/docs/historical-weather-api, `daily=wind_speed_10m_max`, `timezone=America/Los_Angeles`).

**History.** The 2024 version of this column (used in the submitted thesis) was downloaded with a bug:
`date = df["ex_date"][0]` used the first fire's date (2013-02-24) for every row, so it depended only on location.
It was corrected in November 2026. The 2024 file is kept as
`old/data/prescribed_treatment_data_new_clean_2024_buggy_wind.csv`.

**Cache.** Downloads are stored in `data/raw/open_meteo/max_wind_fire_date.csv`, one row per (lat, lon, date),
written as they arrive, so an interrupted run resumes where it stopped. With the cache present no web requests are made.
Set `REFRESH_WIND = True` to download again (about 2,000 requests, ~10 min). Open-Meteo revises its archive from time
to time, so a refresh can change values slightly.

In [12]:
import time
import requests

REFRESH_WIND = False
WIND_CACHE = os.path.join("data", "raw", "open_meteo", "max_wind_fire_date.csv")
os.makedirs(os.path.dirname(WIND_CACHE), exist_ok=True)
session = requests.Session()   # keep-alive: ~0.2 s per request (new connections per request were much slower and stalled)

def fetch_max_wind(lat, lon, date, retries=8):
    """Daily max 10 m wind speed (km/h) on local date `date` (YYYY-MM-DD) at (lat, lon), from the Open-Meteo archive."""
    params = dict(latitude=lat, longitude=lon, start_date=date, end_date=date, daily="wind_speed_10m_max",
                  wind_speed_unit="kmh", timezone="America/Los_Angeles")
    for attempt in range(retries):
        try:
            r = session.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=20)
            if r.status_code == 429:                       # rate limited (600 requests/minute on the free tier)
                time.sleep(61); continue
            r.raise_for_status()
            return r.json()["daily"]["wind_speed_10m_max"][0]
        except requests.RequestException:
            time.sleep(min(2 ** attempt, 60))
    raise RuntimeError(f"failed after {retries} tries: {lat}, {lon}, {date}")

# local (Pacific) calendar date of ignition; ex_date is stored in UTC
tdf["fire_date_local"] = pd.to_datetime(tdf["ex_date"], utc=True).dt.tz_convert("America/Los_Angeles").dt.strftime("%Y-%m-%d")
queries = tdf[["incident_latitude", "incident_longitude", "fire_date_local"]].drop_duplicates()

if REFRESH_WIND and os.path.exists(WIND_CACHE):
    os.remove(WIND_CACHE)
cache = pd.read_csv(WIND_CACHE) if os.path.exists(WIND_CACHE) else pd.DataFrame(columns=list(queries.columns) + ["max_wind"])
done = set(map(tuple, cache[list(queries.columns)].astype(str).values))
todo = [q for q in queries.itertuples(index=False) if tuple(map(str, q)) not in done]
print(f"{len(queries)} unique (location, date) queries; {len(todo)} to download")
if todo:
    write_header = not os.path.exists(WIND_CACHE)
    with open(WIND_CACHE, "a") as fh:
        if write_header:
            fh.write("incident_latitude,incident_longitude,fire_date_local,max_wind\n")
        for q in tqdm(todo):
            w = fetch_max_wind(q.incident_latitude, q.incident_longitude, q.fire_date_local)
            fh.write(f"{q.incident_latitude!r},{q.incident_longitude!r},{q.fire_date_local},{w}\n"); fh.flush()
            time.sleep(0.1)   # stay under 600 requests/minute


1988 unique (location, date) queries; 0 to download


In [13]:
cache = pd.read_csv(WIND_CACHE).drop_duplicates(["incident_latitude", "incident_longitude", "fire_date_local"])
tdf = tdf.drop(columns=["max_wind"], errors="ignore").merge(
    cache, on=["incident_latitude", "incident_longitude", "fire_date_local"], how="left", validate="many_to_one")
tdf = tdf.drop(columns="fire_date_local")
print("missing max_wind:", tdf["max_wind"].isna().sum())
tdf.to_csv(os.path.join(OUT_DIR, "prescribed_treatment_data_rebuilt.csv"))
tdf["max_wind"].describe().round(2)

missing max_wind: 0


count    1992.00
mean       16.48
std         6.40
min         4.10
25%        12.30
50%        15.00
75%        19.20
max        51.00
Name: max_wind, dtype: float64

## Check the rebuild against the canonical thesis files

In [14]:
canon = pd.read_csv("data/processed/prescribed_treatment_data_new_clean.csv", index_col=0)
rebuilt = pd.read_csv(os.path.join(OUT_DIR, "prescribed_treatment_data_rebuilt.csv"), index_col=0)
print("canonical:", canon.shape, " rebuilt:", rebuilt.shape)
shared = [c for c in canon.columns if c in rebuilt.columns]
print("columns only in canonical:", [c for c in canon.columns if c not in rebuilt.columns])
print("columns only in rebuilt:  ", [c for c in rebuilt.columns if c not in canon.columns])
if len(canon) == len(rebuilt):
    mismatch = {}
    for c in shared:
        a, b = canon[c], rebuilt[c]
        if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b) and not pd.api.types.is_bool_dtype(a):
            bad = ~np.isclose(a.astype(float), b.astype(float), rtol=1e-6, atol=1e-6, equal_nan=True)
        else:
            bad = a.astype(str).values != b.astype(str).values
        if bad.sum(): mismatch[c] = int(bad.sum())
    print("columns with differing values:", mismatch if mismatch else "none")

canonical: (1992, 82)  rebuilt: (1992, 82)
columns only in canonical: []
columns only in rebuilt:   []
columns with differing values: none


In [15]:
m_canon = pd.read_csv("data/processed/monthly_data.csv", index_col=0)
m_rebuilt = pd.read_csv(os.path.join(OUT_DIR, "monthly_data_rebuilt.csv"), index_col=0)
print(m_canon.shape, m_rebuilt.shape)
print("max abs difference per column:")
print((m_canon.reset_index(drop=True)[m_canon.columns] - m_rebuilt.reset_index(drop=True)[m_canon.columns]).abs().max())

(120, 16) (120, 16)
max abs difference per column:
rx_acres                 0.000003
rx_acres_3_month         0.000005
rx_acres_6_month         0.000005
rx_acres_9_month         0.000005
rx_acres_12_month        0.000045
incident_acres_burned    0.000000
cdd                      0.000000
hdd                      0.000000
pcp                      0.000000
pdsi                     0.000000
phdi                     0.000000
pmdi                     0.000000
tavg                     0.000000
tmax                     0.000000
tmin                     0.000000
zndx                     0.000000
dtype: float64
